# Plot the output of cellpose_GFP_RFP

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.3 

settheme <- theme_minimal() +
  theme(
    text = element_text(family = "sans", size = FONT.SIZE),
    panel.background = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(colour = "black"),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 0),
    legend.position = "right",
    title = element_text(colour = "black", size = FONT.SIZE),
    plot.title = element_text(size = FONT.SIZE, face = "plain")
  )

In [ ]:
col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")
col_GFP = "#86AB30"
col_RFP = "#EB5951"

col_condition = c("Developed" = "#285F62", 
               "Failed" = "#CA4F33")


## 1. Extract summary files

In [ ]:
root_dir <- "/ceph.groups/mshahbazi.grp/rsakata/EXP49/output/batch3/cellpose_intensities"
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP49/output/batch3/plots"
EXP = "EXP49"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
}


# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind; adds a 'source_file' column with the full path
combined_df <- csv_paths |>
  set_names(basename(csv_paths)) |>
  purrr::map_dfr(\(p) readr::read_csv(p, show_col_types = FALSE), .id = "source_file")


combined_df$sample <- sub("_cells_with_spheroid\\.csv$", "", combined_df$source_file)

In [ ]:
head(combined_df)

In [ ]:
combined_df <- combined_df %>%
  rename(image = sample) %>%
  mutate(sample = str_extract(image, "(?<=mcherry_)[^-]+"))

combined_df$sample <- as.character(combined_df$sample)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv("/ceph.groups/mshahbazi.grp/rsakata/EXP49/sample_sheet.csv", show_col_types = FALSE)

sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- combined_df %>%
  left_join(sample_sheet, by = "sample")  

# join in sample in sample sheet is contained in combined df sample
# merged_df <- regex_left_join(
#   combined_df,
#   sample_sheet,
#   by = c("sample"),
#   ignore_case = FALSE
# ) 

In [ ]:
head(merged_df)

In [ ]:
merged_df <- merged_df %>%
  rename(
    'GFP_intensity' = MeanIntensity_C2,
    'RFP_intensity' = MeanIntensity_C3
  )

In [ ]:
unique(merged_df$sample_name)

In [ ]:
tbl <- merged_df %>%
  group_by(sample, sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange((sample))  # optional

tbl

## a) Find thresholds

In [ ]:
title = "GFP_RFP_scatter"
SLOPE = 1

w <- 4
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = GFP_intensity, y = RFP_intensity, color = sample_name)) +
  geom_point(alpha = 0.6, size = 0.2) +
  labs(x = "GFP intensity", y = "RFP intensity", title = "") +
  settheme + 
  scale_x_continuous(breaks = seq(0, 250, by = 100))+
  #scale_color_manual(values=col_condition)+
  geom_abline(slope = SLOPE, intercept = 0, linetype = "dashed", color = "grey20") +
  coord_equal(ratio = 1)+ 
  facet_grid(timepoint~condition)+
  theme(legend.position = "none")
ggsave(plot = ggscatter, filename = sprintf("%s/a_%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
ggplot(merged_df, aes(x = GFP_intensity)) +
  geom_histogram(bins = 30, color = "black", fill = "#06ce31ff", na.rm = TRUE) +
  labs(x = "GFP intensity", y = "Count", title = "GFP intensity histogram") +
  settheme+
  geom_vline(xintercept = 50, linetype = "dashed") 

In [ ]:
ggplot(merged_df, aes(x = RFP_intensity)) +
  geom_histogram(bins = 30, color = "black", fill = "#da2a82ff", na.rm = TRUE) +
  labs(x = "RFP intensity", y = "Count", title = "RFP intensity histogram") +
  settheme+
  geom_vline(xintercept = 50, linetype = "dashed") 

## Relabel

In [ ]:
library(dplyr)

merged_df <- merged_df %>%
  mutate(
    label = case_when(
      GFP_intensity < 50 & RFP_intensity < 50 ~ "No_label",
      (RFP_intensity / GFP_intensity) > 1   & RFP_intensity >100     ~ "RFP",   # slope = y/x > 1
      TRUE                                      ~ "GFP"
    )
  )


In [ ]:
head(merged_df)

## Reformat and process for each sample

In [ ]:
colnames(merged_df)

In [ ]:
df_sample <- merged_df %>%
  group_by(sample_name, image,condition,timepoint, state , condition_2 ) %>%
  summarise(
    # metrics
    average_vol_um3 = mean(Volume_um3, na.rm = TRUE),
    total_cells     = n(),
    RFPpos          = sum(label == "RFP", na.rm = TRUE),
    GFPpos          = sum(label == "GFP", na.rm = TRUE),
    Neg          = sum(label == "No_label", na.rm = TRUE),
    propRFP         = RFPpos / total_cells,
    propGFP         = GFPpos / total_cells,
    propNeg         = Neg / total_cells,
    .groups = "drop"
  )


In [ ]:
head(df_sample)

In [ ]:
merged_df = merged_df %>%
  filter(state == 'Failed')

In [ ]:
order_cond <- c("G_R","Grev_Rrev",  "Grev_R", "Rrev_G")

df_sample <- df_sample %>%
  mutate(condition = factor(condition, levels = order_cond))

In [ ]:
unique(df_sample$sample_name)

# Plot 

In [ ]:
df_sample <- merged_df %>%
  group_by(sample_name, image,condition,timepoint, state ) %>%
  summarise(
    # metrics
    average_vol_um3 = mean(Volume_um3, na.rm = TRUE),
    total_labelled     = sum(label %in% c("RFP", "GFP"), na.rm = TRUE),
    RFPpos          = sum(label == "RFP", na.rm = TRUE),
    GFPpos          = sum(label == "GFP", na.rm = TRUE),
    Neg          = sum(label == "No_label", na.rm = TRUE),
    propRFP         = RFPpos / total_labelled,
    propGFP         = GFPpos / total_labelled,
    .groups = "drop"
  )

order_cond <- c("G_R","Grev_Rrev",  "Grev_R", "Rrev_G")

df_sample <- df_sample %>%
  mutate(condition = factor(condition, levels = order_cond))

In [ ]:
unique(df_sample$condition)

In [ ]:
head(df_sample)

### A) total cell numbers

In [ ]:
title = "counts_per_structure"
w <- 1.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =total_labelled, group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap(~state) +
      scale_color_manual(values=col_condition)+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "counts_per_structure_D4_labelled"
w <- 2
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D4")
  
p = ggplot(df_sample_sub, aes(x = condition, y =total_labelled, group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      aes(color = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = "Total number of cells per structure",
      y = "number of cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
      scale_y_continuous(limits = c(0, 250), expand = c(0, 0))
     # scale_color_manual(values=col_condition)

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### B) Percentage GFP+

In [ ]:
title = "per_GFPpos"
w <- 1.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =propGFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP 
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%EGFP+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("B_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_GFPpos_D6"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =propGFP , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%EGFP+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### C) GFP+ counts

In [ ]:
title = "count_GFPpos"
w <- 1.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

  
p = ggplot(df_sample, aes(x = condition, y =GFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of EGFP+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap( ~ state) 


ggsave(file.path(out_dir, sprintf("C_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "count_GFPpos_D4"
w <- 3
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

df_sample_D4 = df_sample %>% subset(timepoint == "D4")
  
p = ggplot(df_sample_D4, aes(x = condition, y =GFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of EGFP+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "count_GFPpos_labelled_D6"
w <- 3
h <- 2.2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =GFPpos , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of EGFP+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 470), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### D) RFP+ 

In [ ]:
title = "per_RFPpos"
w <- 1.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =propRFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP 
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%mcherry+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_RFPpos_labelled_D6"
w <- 4
h <- 2.2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =propRFP , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%mcherry+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### E) RFP + counts

In [ ]:
title = "count_RFPpos"
w <- 1.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

  
p = ggplot(df_sample, aes(x = condition, y =RFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap( ~ state) 


ggsave(file.path(out_dir, sprintf("E_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "count_RFPpos_labelled_D4"
w <- 3
h <- 2.5
options(repr.plot.width=w, repr.plot.height=h)

df_sample_D4 = df_sample %>% subset(timepoint == "D4")
  
p = ggplot(df_sample_D4, aes(x = condition, y =RFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "count_RFPpos_labelled_D6"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =RFPpos , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### Stacked

In [ ]:
# 1) Summarise to get mean propRFP (and SE) per panel
df_bar <- df_sample %>%
  group_by(condition, state, timepoint) %>%
  summarise(
    mean_propRFP = mean(propRFP, na.rm = TRUE),
    n = sum(!is.na(propRFP)),
    se = sd(propRFP, na.rm = TRUE) / sqrt(pmax(n, 1)),
    .groups = "drop"
  ) %>%
  mutate(remainder = pmax(0, 1 - mean_propRFP))

# 2) Long format for stacked bar
bar_long <- df_bar %>%
  pivot_longer(c(mean_propRFP, remainder),
               names_to = "part", values_to = "value")%>%
  mutate(part = factor(part, levels = c("remainder", "mean_propRFP")))

# 3) Plot
title <- "per_labels_stacked"
w <- 6.5;
h <- 3
options(repr.plot.width = w, repr.plot.height = h)

p <- ggplot() +
  # stacked bar to total 1: bottom = mean_propRFP (grey), top = remainder (green)
  geom_col(
    data = bar_long,
    aes(x = condition, y = value, fill = part),
    width = 0.6, position = "stack", alpha = 0.8
  ) +
  # error bar only for the mean RFP+ portion
  geom_errorbar(
    data = df_bar,
    aes(x = condition,
        ymin = pmax(0, mean_propRFP - se),
        ymax = pmin(1, mean_propRFP + se)),
    width = 0.2, color = "black", position = position_identity()
  ) +
  # jitter of individual points for transparency of the raw distribution
  geom_jitter(
    data = df_sample,
    aes(x = condition, y = propRFP, fill = condition),
    position = position_jitter(width = 0.15, height = 0),
    size = 0.5, alpha = 0.8, color = "#474747ff"
  ) +
  facet_wrap(state ~ timepoint) +
  scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  scale_fill_manual(
    values = c(mean_propRFP = "#E94F37", remainder = "#59be83ff"),
    labels = c(mean_propRFP = "mcherry+", remainder = "EGFP+")
  ) +
  labs(
    title = title,
    x = "condition",
    y = "%mCherry+"
  ) +
  settheme +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.title = element_blank())

p

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
       plot = p, width = w, height = h)


In [ ]:
# 3) Plot
title <- "per_labels_stacked_D6"
w <- 6;
h <- 3
options(repr.plot.width = w, repr.plot.height = h)


# 1) Summarise to get mean propRFP (and SE) per panel
df_bar <- df_sample_sub  %>%
  group_by(condition, state) %>%
  summarise(
    mean_propRFP = mean(propRFP, na.rm = TRUE),
    n = sum(!is.na(propRFP)),
    se = sd(propRFP, na.rm = TRUE) / sqrt(pmax(n, 1)),
    .groups = "drop"
  ) %>%
  mutate(remainder = pmax(0, 1 - mean_propRFP))

# 2) Long format for stacked bar
bar_long <- df_bar %>%
  pivot_longer(c(mean_propRFP, remainder),
               names_to = "part", values_to = "value")%>%
  mutate(part = factor(part, levels = c("remainder", "mean_propRFP")))


p <- ggplot() +
  # stacked bar to total 1: bottom = mean_propRFP (grey), top = remainder (green)
  geom_col(
    data = bar_long,
    aes(x = state , y = value, fill = part),
    width = 0.6, position = "stack", alpha = 0.8
  ) +
  # error bar only for the mean RFP+ portion
  geom_errorbar(
    data = df_bar,
    aes(x = state ,
        ymin = pmax(0, mean_propRFP - se),
        ymax = pmin(1, mean_propRFP + se)),
    width = 0.2, color = "black", position = position_identity()
  ) +
  # jitter of individual points for transparency of the raw distribution
  geom_jitter(
    data = df_sample_sub,
    aes(x = state , y = propRFP, fill = condition),
    position = position_jitter(width = 0.15, height = 0),
    size = 0.5, alpha = 0.8, color = "#474747ff"
  ) +
  facet_wrap(~condition, nrow = 1) +
  scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  scale_fill_manual(
    values = c(mean_propRFP = "#E94F37", remainder = "#59be83ff"),
    labels = c(mean_propRFP = "mcherry+", remainder = "EGFP+")
  ) +
  labs(
    title = title,
    x = "condition",
    y = "%mCherry+"
  ) +
  settheme +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.title = element_blank())

p

ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
       plot = p, width = w, height = h)

# Save

In [ ]:
df_sample$EXP = EXP

In [ ]:
write_csv(df_sample, file.path(out_dir, "summarised_results.csv"))